# UR5 Reach: Codebase and Library Notes

这份笔记更偏“学习笔记”，结合本项目代码解释常用库和 Python 语法用法。
可以当作阅读 `ur5_reach_env.py` 和 `train_ur5_reach.py` 的对照指南。

## 1. Python 基础语法在项目中的使用

- `Path(__file__).resolve()`：获取当前文件绝对路径
- `f"...{var}..."`：f-string 格式化
- `np.array([...], dtype=...)`：显式指定数组类型
- `for _ in range(n)`：只关心次数时用 `_`
- `if value is None`：判断空值或未初始化状态

In [ ]:
from pathlib import Path

current_file = Path(__file__).resolve() if "__file__" in globals() else Path(".").resolve()
print("current:", current_file)
name = "ur5"
print(f"robot={name}")

## 2. NumPy 与观测拼接

环境里常见的操作是把多个向量拼接成一个观测：
- 目标相对位置
- 关节角
- 关节速度
- 上一步扭矩
- 末端速度

In [ ]:
import numpy as np

relative_pos = np.zeros(3)
joint_angles = np.zeros(6)
joint_vel = np.zeros(6)
prev_torque = np.zeros(6)
ee_vel = np.zeros(3)

obs = np.concatenate([relative_pos, joint_angles, joint_vel, prev_torque, ee_vel])
print("obs shape:", obs.shape)

## 3. Gymnasium 环境 API

Gymnasium 环境最核心的三个方法：
- `reset()`：返回初始观测
- `step(action)`：返回 `(obs, reward, done, truncated, info)`
- `render()`：可视化（`render_mode='human'`）

In [ ]:
from ur5_reach_env import UR5ReachEnv

env = UR5ReachEnv(render_mode=None)
obs, info = env.reset()
action = env.action_space.sample()
obs, reward, done, truncated, info = env.step(action)
print("reward:", reward)
env.close()

## 4. MuJoCo 的 Model / Data 结构

MuJoCo 使用 `MjModel` 保存“结构”，`MjData` 保存“状态”。
常见访问方式：
- `data.qpos` / `data.qvel`：关节位置与速度
- `data.ctrl`：控制输入
- `data.xpos`：各个 body 的世界坐标
- `mj_name2id(...)`：名字查索引

## 5. Stable-Baselines3 的使用思路

主线训练脚本里核心步骤通常是：
1. 创建环境（必要时用 `VecNormalize`）
2. 创建算法（SAC/TD3/PPO）
3. `model.learn(...)` 开始训练
4. 保存模型和归一化参数

In [ ]:
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv
from ur5_reach_env import UR5ReachEnv

env = DummyVecEnv([lambda: UR5ReachEnv(render_mode=None)])
model = SAC("MlpPolicy", env, verbose=1)
print("ready to train")

## 6. 读代码时的重点

- `UR5ReachEnv.step()`：奖励函数的每一项都在这里
- `TrainRenderCallback`：训练时如何渲染
- `VecNormalize`：为什么训练时归一化、测试时要关掉
- `EvalCallback`：评估与保存最佳模型